In [ ]:
import pandas as pd

stats = pd.read_csv("player_match_stats_v2.csv")
df = stats.copy()

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1996 entries, 0 to 1995
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   match_id         1996 non-null   int64  
 1   player_id        1996 non-null   float64
 2   player           1996 non-null   object 
 3   team_id          1996 non-null   int64  
 4   team             1996 non-null   object 
 5   passes           1996 non-null   int64  
 6   shots            1996 non-null   int64  
 7   carries          1996 non-null   int64  
 8   pressures        1996 non-null   int64  
 9   ball_recoveries  1996 non-null   int64  
 10  duels            1996 non-null   int64  
 11  clearances       1996 non-null   int64  
 12  blocks           1996 non-null   int64  
 13  dribbles         1996 non-null   int64  
 14  interceptions    1996 non-null   int64  
 15  fouls_committed  1996 non-null   int64  
 16  fouls_won        1996 non-null   int64  
 17  dispossessed  

Based on the `df.info()` output, I will identify the numerical columns that represent player statistics. I'll exclude `match_id`, `player_id`, `player`, `team_id`, and `team` from the ranking calculation, as these are identifiers or categorical features.

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns
# Exclude identifier columns from ranking
columns_to_exclude = ['match_id', 'player_id', 'team_id']
stats_cols = [col for col in numeric_cols if col not in columns_to_exclude]

print(f"Columns selected for ranking: {stats_cols}")

Columns selected for ranking: ['passes', 'shots', 'carries', 'pressures', 'ball_recoveries', 'duels', 'clearances', 'blocks', 'dribbles', 'interceptions', 'fouls_committed', 'fouls_won', 'dispossessed', 'miscontrols', 'pass_accuracy', 'goals', 'shots_on_target', 'xG']


In [ ]:
# Calculate percentile rank for each statistic within each match
for col in stats_cols:
    rank_col_name = f"{col}_rank_percentile"
    df[rank_col_name] = df.groupby('match_id')[col].rank(pct=True)

# Drop the original statistical columns, keeping only identifiers and rank percentiles
df_per_match = df.drop(columns=stats_cols)
display(df_per_match.head())

,match_id,player_id,player,team_id,team,passes_rank_percentile,shots_rank_percentile,carries_rank_percentile,pressures_rank_percentile,ball_recoveries_rank_percentile,...,dribbles_rank_percentile,interceptions_rank_percentile,fouls_committed_rank_percentile,fouls_won_rank_percentile,dispossessed_rank_percentile,miscontrols_rank_percentile,pass_accuracy_rank_percentile,goals_rank_percentile,shots_on_target_rank_percentile,xG_rank_percentile
0,3857254,3043.0,Christian Dannemann Eriksen,776,Denmark,0.967742,0.677419,0.951613,0.709677,0.709677,...,1.000000,0.290323,0.887097,0.290323,0.854839,0.870968,0.580645,0.516129,0.919355,0.806452
1,3857254,3557.0,Naïm Sliti,777,Tunisia,0.225806,0.274194,0.225806,0.225806,0.564516,...,0.838710,0.677419,0.274194,0.709677,0.274194,0.870968,0.096774,0.516129,0.435484,0.274194
2,3857254,3570.0,Pierre-Emile Højbjerg,776,Denmark,0.903226,0.677419,0.951613,0.612903,0.806452,...,0.612903,0.677419,0.274194,0.709677,0.951613,0.629032,0.903226,0.516129,0.919355,0.548387
3,3857254,3767.0,Ellyes Joris Skhiri,777,Tunisia,0.580645,0.870968,0.483871,1.000000,1.000000,...,0.612903,0.290323,0.661290,0.709677,0.274194,0.241935,0.774194,0.516129,0.435484,0.838710
4,3857254,3815.0,Kasper Schmeichel,776,Denmark,0.451613,0.274194,0.274194,0.048387,0.919355,...,0.241935,0.290323,0.274194,0.290323,0.274194,0.241935,0.403226,0.516129,0.435484,0.274194


Now, let's calculate the percentile rank for each statistic across the *entire dataset* (all matches combined). This will give us a global ranking for each player's performance in a given statistical category.

In [ ]:
# Create a copy of the original DataFrame to avoid modifying it directly
df_whole = stats[['match_id', 'player_id', 'player', 'team_id', 'team']].copy()

# Calculate global percentile rank for each statistic across the entire DataFrame
for col in stats_cols:
    rank_col_name_global = f"{col}_global_rank_percentile"
    df_whole[rank_col_name_global] = df[col].rank(pct=True)

# Display the DataFrame with the new global percentile rank columns
display(df_whole.head())

,match_id,player_id,player,team_id,team,passes_global_rank_percentile,shots_global_rank_percentile,carries_global_rank_percentile,pressures_global_rank_percentile,ball_recoveries_global_rank_percentile,...,dribbles_global_rank_percentile,interceptions_global_rank_percentile,fouls_committed_global_rank_percentile,fouls_won_global_rank_percentile,dispossessed_global_rank_percentile,miscontrols_global_rank_percentile,pass_accuracy_global_rank_percentile,goals_global_rank_percentile,shots_on_target_global_rank_percentile,xG_global_rank_percentile
0,3857254,3043.0,Christian Dannemann Eriksen,776,Denmark,0.942886,0.706413,0.959419,0.651553,0.719439,...,0.973196,0.288828,0.829409,0.261523,0.880010,0.835421,0.556112,0.463427,0.884018,0.799098
1,3857254,3557.0,Naïm Sliti,777,Tunisia,0.185621,0.290080,0.330661,0.241232,0.578657,...,0.845942,0.704158,0.248246,0.654309,0.287575,0.835421,0.120992,0.463427,0.404309,0.290080
2,3857254,3570.0,Pierre-Emile Højbjerg,776,Denmark,0.916333,0.706413,0.959419,0.596693,0.822395,...,0.675852,0.704158,0.248246,0.654309,0.951152,0.646543,0.847194,0.463427,0.884018,0.589178
3,3857254,3767.0,Ellyes Joris Skhiri,777,Tunisia,0.594439,0.881764,0.572144,0.952906,0.974699,...,0.675852,0.288828,0.625501,0.654309,0.287575,0.259269,0.654058,0.463427,0.404309,0.819138
4,3857254,3815.0,Kasper Schmeichel,776,Denmark,0.495240,0.290080,0.379008,0.045591,0.929359,...,0.280311,0.288828,0.248246,0.261523,0.287575,0.259269,0.374499,0.463427,0.404309,0.290080


### Player Impact Score Model

To create a player impact score, we'll combine the per-match and global percentile ranks. We need to identify which statistics are 'positive' (higher rank is better) and 'negative' (lower rank is better). For 'negative' statistics, we'll invert their rank (e.g., `1 - rank`) so that a higher value always contributes positively to the overall impact score.

Then, we'll average all these adjusted rank percentiles to get a composite impact score for each player in each match.

In [ ]:
# Merge df_per_match and df_whole to combine all percentile ranks
# We'll need to handle the player, team, match IDs carefully during merge
# First, let's ensure unique identifiers for merging

# df_per_match already has suffix '_rank_percentile'
# df_whole already has suffix '_global_rank_percentile'

# Ensure both DataFrames have the same identifier columns to merge on
# Select common identifier columns
id_cols = ['match_id', 'player_id', 'player', 'team_id', 'team']

# It's generally safer to merge on unique keys. For player stats in a match,
# (match_id, player_id) is usually unique. Player and team names are metadata.

# Let's clean up column names in df_per_match if they conflict after merge
# (though they shouldn't with the chosen merge strategy below)

merged_df = pd.merge(df_per_match, df_whole, on=id_cols, how='left',
                     suffixes=('_per_match', '_global'))

# Display a sample of the merged DataFrame to check column names
display(merged_df.head())

,match_id,player_id,player,team_id,team,passes_rank_percentile,shots_rank_percentile,carries_rank_percentile,pressures_rank_percentile,ball_recoveries_rank_percentile,...,dribbles_global_rank_percentile,interceptions_global_rank_percentile,fouls_committed_global_rank_percentile,fouls_won_global_rank_percentile,dispossessed_global_rank_percentile,miscontrols_global_rank_percentile,pass_accuracy_global_rank_percentile,goals_global_rank_percentile,shots_on_target_global_rank_percentile,xG_global_rank_percentile
0,3857254,3043.0,Christian Dannemann Eriksen,776,Denmark,0.967742,0.677419,0.951613,0.709677,0.709677,...,0.973196,0.288828,0.829409,0.261523,0.880010,0.835421,0.556112,0.463427,0.884018,0.799098
1,3857254,3557.0,Naïm Sliti,777,Tunisia,0.225806,0.274194,0.225806,0.225806,0.564516,...,0.845942,0.704158,0.248246,0.654309,0.287575,0.835421,0.120992,0.463427,0.404309,0.290080
2,3857254,3570.0,Pierre-Emile Højbjerg,776,Denmark,0.903226,0.677419,0.951613,0.612903,0.806452,...,0.675852,0.704158,0.248246,0.654309,0.951152,0.646543,0.847194,0.463427,0.884018,0.589178
3,3857254,3767.0,Ellyes Joris Skhiri,777,Tunisia,0.580645,0.870968,0.483871,1.000000,1.000000,...,0.675852,0.288828,0.625501,0.654309,0.287575,0.259269,0.654058,0.463427,0.404309,0.819138
4,3857254,3815.0,Kasper Schmeichel,776,Denmark,0.451613,0.274194,0.274194,0.048387,0.919355,...,0.280311,0.288828,0.248246,0.261523,0.287575,0.259269,0.374499,0.463427,0.404309,0.290080


In [ ]:
# Define 'positive' and 'negative' statistics based on domain knowledge
# A higher percentile for a 'positive' stat is good.
# A higher percentile for a 'negative' stat is bad, so we'll invert it.

positive_stats_base = [
    'passes', 'shots', 'carries', 'pressures', 'ball_recoveries', 'duels',
    'clearances', 'blocks', 'dribbles', 'interceptions', 'fouls_won',
    'completed_passes', 'pass_accuracy', 'goals', 'shots_on_target', 'xG'
]

negative_stats_base = [
    'fouls_committed', 'dispossessed', 'miscontrols'
]

# Create lists of the actual column names in merged_df for these stats
positive_rank_cols = []
negative_rank_cols = []

for stat in positive_stats_base:
    if f'{stat}_rank_percentile' in merged_df.columns:
        positive_rank_cols.append(f'{stat}_rank_percentile')
    if f'{stat}_global_rank_percentile' in merged_df.columns:
        positive_rank_cols.append(f'{stat}_global_rank_percentile')

for stat in negative_stats_base:
    if f'{stat}_rank_percentile' in merged_df.columns:
        negative_rank_cols.append(f'{stat}_rank_percentile')
    if f'{stat}_global_rank_percentile' in merged_df.columns:
        negative_rank_cols.append(f'{stat}_global_rank_percentile')

print(f"Positive rank columns: {positive_rank_cols}")
print(f"Negative rank columns: {negative_rank_cols}")

# Invert percentile ranks for 'negative' statistics
for col in negative_rank_cols:
    merged_df[col] = 1 - merged_df[col]

display(merged_df.head())

Positive rank columns: ['passes_rank_percentile', 'passes_global_rank_percentile', 'shots_rank_percentile', 'shots_global_rank_percentile', 'carries_rank_percentile', 'carries_global_rank_percentile', 'pressures_rank_percentile', 'pressures_global_rank_percentile', 'ball_recoveries_rank_percentile', 'ball_recoveries_global_rank_percentile', 'duels_rank_percentile', 'duels_global_rank_percentile', 'clearances_rank_percentile', 'clearances_global_rank_percentile', 'blocks_rank_percentile', 'blocks_global_rank_percentile', 'dribbles_rank_percentile', 'dribbles_global_rank_percentile', 'interceptions_rank_percentile', 'interceptions_global_rank_percentile', 'fouls_won_rank_percentile', 'fouls_won_global_rank_percentile', 'pass_accuracy_rank_percentile', 'pass_accuracy_global_rank_percentile', 'goals_rank_percentile', 'goals_global_rank_percentile', 'shots_on_target_rank_percentile', 'shots_on_target_global_rank_percentile', 'xG_rank_percentile', 'xG_global_rank_percentile']
Negative rank c

,match_id,player_id,player,team_id,team,passes_rank_percentile,shots_rank_percentile,carries_rank_percentile,pressures_rank_percentile,ball_recoveries_rank_percentile,...,dribbles_global_rank_percentile,interceptions_global_rank_percentile,fouls_committed_global_rank_percentile,fouls_won_global_rank_percentile,dispossessed_global_rank_percentile,miscontrols_global_rank_percentile,pass_accuracy_global_rank_percentile,goals_global_rank_percentile,shots_on_target_global_rank_percentile,xG_global_rank_percentile
0,3857254,3043.0,Christian Dannemann Eriksen,776,Denmark,0.967742,0.677419,0.951613,0.709677,0.709677,...,0.973196,0.288828,0.170591,0.261523,0.119990,0.164579,0.556112,0.463427,0.884018,0.799098
1,3857254,3557.0,Naïm Sliti,777,Tunisia,0.225806,0.274194,0.225806,0.225806,0.564516,...,0.845942,0.704158,0.751754,0.654309,0.712425,0.164579,0.120992,0.463427,0.404309,0.290080
2,3857254,3570.0,Pierre-Emile Højbjerg,776,Denmark,0.903226,0.677419,0.951613,0.612903,0.806452,...,0.675852,0.704158,0.751754,0.654309,0.048848,0.353457,0.847194,0.463427,0.884018,0.589178
3,3857254,3767.0,Ellyes Joris Skhiri,777,Tunisia,0.580645,0.870968,0.483871,1.000000,1.000000,...,0.675852,0.288828,0.374499,0.654309,0.712425,0.740731,0.654058,0.463427,0.404309,0.819138
4,3857254,3815.0,Kasper Schmeichel,776,Denmark,0.451613,0.274194,0.274194,0.048387,0.919355,...,0.280311,0.288828,0.751754,0.261523,0.712425,0.740731,0.374499,0.463427,0.404309,0.290080


In [ ]:
# Combine all rank columns (positive and adjusted negative) for the score calculation
all_impact_cols = positive_rank_cols + negative_rank_cols
print(f"All impact columns identified: {len(all_impact_cols)} columns")

All impact columns identified: 36 columns


In [ ]:
# Define offensive and defensive base statistics (from the original stats_cols)
offensive_stats_base = [
    'passes', 'shots', 'carries', 'dribbles', 'goals', 'shots_on_target', 'xG', 'pass_accuracy',
    'fouls_won', 'dispossessed', 'miscontrols'
]
defensive_stats_base = [
    'pressures', 'ball_recoveries', 'duels', 'clearances', 'blocks', 'interceptions',
    'fouls_committed'
]

# Create lists of the actual rank columns (both per-match and global) for offensive and defensive categories
offensive_rank_cols = []
defensive_rank_cols = []

for stat in offensive_stats_base:
    # Check for both per-match and global ranks in the `all_impact_cols` (which contains adjusted ranks)
    per_match_col = f'{stat}_rank_percentile'
    global_col = f'{stat}_global_rank_percentile'
    if per_match_col in all_impact_cols:
        offensive_rank_cols.append(per_match_col)
    if global_col in all_impact_cols:
        offensive_rank_cols.append(global_col)

for stat in defensive_stats_base:
    per_match_col = f'{stat}_rank_percentile'
    global_col = f'{stat}_global_rank_percentile'
    if per_match_col in all_impact_cols:
        defensive_rank_cols.append(per_match_col)
    if global_col in all_impact_cols:
        defensive_rank_cols.append(global_col)

print(f"Offensive rank columns identified: {len(offensive_rank_cols)} columns")
print(f"Defensive rank columns identified: {len(defensive_rank_cols)} columns")

Offensive rank columns identified: 22 columns
Defensive rank columns identified: 14 columns


In [ ]:
# Calculate the overall Player Impact Score by averaging all relevant percentile ranks
# `all_impact_cols` is already defined and contains positive and inverted negative rank columns
merged_df['player_impact_score'] = merged_df[all_impact_cols].mean(axis=1)

# Define offensive and defensive base statistics (from the original stats_cols)
# These lists are already in the kernel state

# Calculate average offensive and defensive ranks within merged_df (per player per match)
merged_df['avg_offensive_rank'] = merged_df[offensive_rank_cols].mean(axis=1)
merged_df['avg_defensive_rank'] = merged_df[defensive_rank_cols].mean(axis=1)

# Display a sample of merged_df with the newly calculated scores
display(merged_df[['match_id', 'player', 'player_impact_score', 'avg_offensive_rank', 'avg_defensive_rank']].head())

,match_id,player,player_impact_score,avg_offensive_rank,avg_defensive_rank
0,3857254,Christian Dannemann Eriksen,0.546161,0.627933,0.417662
1,3857254,Naïm Sliti,0.402901,0.405184,0.399313
2,3857254,Pierre-Emile Højbjerg,0.658512,0.648170,0.674765
3,3857254,Ellyes Joris Skhiri,0.666394,0.658138,0.679366
4,3857254,Kasper Schmeichel,0.401590,0.424400,0.365745


In [ ]:
# 1. per-match, all stats
per_match_all_cols = [col for col in all_impact_cols if col.endswith('_rank_percentile') and not col.endswith('_global_rank_percentile')]
merged_df['avg_all_per_match_ranks'] = merged_df[per_match_all_cols].mean(axis=1)

# 2. per-match, offensive stats
per_match_offensive_cols = [col for col in offensive_rank_cols if col.endswith('_rank_percentile') and not col.endswith('_global_rank_percentile')]
merged_df['avg_offensive_per_match_ranks'] = merged_df[per_match_offensive_cols].mean(axis=1)

# 3. per-match, defensive stats
per_match_defensive_cols = [col for col in defensive_rank_cols if col.endswith('_rank_percentile') and not col.endswith('_global_rank_percentile')]
merged_df['avg_defensive_per_match_ranks'] = merged_df[per_match_defensive_cols].mean(axis=1)

# 4. global, all stats
global_all_cols = [col for col in all_impact_cols if col.endswith('_global_rank_percentile')]
merged_df['avg_all_global_ranks'] = merged_df[global_all_cols].mean(axis=1)

# 5. global, offensive stats
global_offensive_cols = [col for col in offensive_rank_cols if col.endswith('_global_rank_percentile')]
merged_df['avg_offensive_global_ranks'] = merged_df[global_offensive_cols].mean(axis=1)

# 6. global, defensive stats
global_defensive_cols = [col for col in defensive_rank_cols if col.endswith('_global_rank_percentile')]
merged_df['avg_defensive_global_ranks'] = merged_df[global_defensive_cols].mean(axis=1)

# Display a sample of merged_df with the new average rank columns
display(merged_df[[
    'match_id', 'player',
    'avg_all_per_match_ranks', 'avg_offensive_per_match_ranks', 'avg_defensive_per_match_ranks',
    'avg_all_global_ranks', 'avg_offensive_global_ranks', 'avg_defensive_global_ranks'
]].head())

,match_id,player,avg_all_per_match_ranks,avg_offensive_per_match_ranks,avg_defensive_per_match_ranks,avg_all_global_ranks,avg_offensive_global_ranks,avg_defensive_global_ranks
0,3857254,Christian Dannemann Eriksen,0.548387,0.634897,0.412442,0.543935,0.620969,0.422881
1,3857254,Naïm Sliti,0.399642,0.404692,0.391705,0.406160,0.405675,0.406921
2,3857254,Pierre-Emile Højbjerg,0.657706,0.651026,0.668203,0.659319,0.645313,0.681327
3,3857254,Ellyes Joris Skhiri,0.668459,0.664223,0.675115,0.664329,0.652054,0.683617
4,3857254,Kasper Schmeichel,0.398746,0.422287,0.361751,0.404434,0.426512,0.369739


In [ ]:
player_impact_summary = merged_df.groupby(['player_id', 'player', 'team']).agg(
    avg_all_per_match_ranks=('avg_all_per_match_ranks', 'mean'),
    avg_offensive_per_match_ranks=('avg_offensive_per_match_ranks', 'mean'),
    avg_defensive_per_match_ranks=('avg_defensive_per_match_ranks', 'mean'),
    avg_all_global_ranks=('avg_all_global_ranks', 'mean'),
    avg_offensive_global_ranks=('avg_offensive_global_ranks', 'mean'),
    avg_defensive_global_ranks=('avg_defensive_global_ranks', 'mean')
).reset_index()

display(player_impact_summary.head())

player_impact_summary.to_csv(
    "player_stats_rank.csv",
    index=False
)

,player_id,player,team,avg_all_per_match_ranks,avg_offensive_per_match_ranks,avg_defensive_per_match_ranks,avg_all_global_ranks,avg_offensive_global_ranks,avg_defensive_global_ranks
0,2941.0,Ismaïla Sarr,Senegal,0.597203,0.599048,0.594303,0.573915,0.575993,0.570650
1,2954.0,Youri Tielemans,Belgium,0.449071,0.430534,0.478200,0.441350,0.439698,0.443947
2,2972.0,Marcus Thuram,France,0.409875,0.385834,0.447653,0.405127,0.384442,0.437632
3,2988.0,Memphis Depay,Netherlands,0.502790,0.545298,0.435990,0.497615,0.541396,0.428815
4,2989.0,Alexander Djiku,Ghana,0.466552,0.447420,0.496616,0.451271,0.442909,0.464411
